In [ ]:
from evalplus.data import get_human_eval_plus
import requests, json, os

problems = get_human_eval_plus()

print(f"Number of problems: {len(problems)}")

ollama_url = "http://host.docker.internal:11434/api/generate"

samples = []

for task_id in problems.keys():
  problem = problems[task_id]
  prompt = problem["prompt"]

  payload = {
    "model": "qwen2.5-coder:3b",
    "prompt": prompt,
    "stream": False
  }
  response = requests.post(ollama_url, json=payload).json()
  generated_code = response.get("response", "")

  samples.append({
    "task_id": task_id,
    "solution": generated_code
  })

path = "../data/processed/"
if not os.path.exists(path):
  os.makedirs(path)
output_file = f"{path}test_output.jsonl"
with open(output_file, "w") as f:
  for sample in samples:
    f.write(json.dumps(sample) + "\n")


The results contained natural language explantations which made the tests fail. So we cleaned it with the following code:

In [ ]:
import re

regex = r"```python\n(.*?)\n```"

outputsamples = []
file = "../data/processed/test_output.jsonl"
with open(file, "r") as f:
  for line in f:
    sample = json.loads(line)
    solution = sample['solution']
    match = re.search(regex, solution, re.DOTALL)
    solution = match.group(1) if match else solution
    sample['solution'] = solution
    outputsamples.append(sample)


with open(file, "w") as f:
  for sample in outputsamples:
    f.write(json.dumps(sample)+"\n")

In [3]:
!evalplus.evaluate --dataset humaneval --samples ../data/processed/test_output.jsonl

Load from ground-truth from /home/vscode/.cache/evalplus/fe585eb4df8c88d844eeb463ea4d0302.pkl
Reading samples...
164it [00:00, 1690.32it/s]
100%|████████████████████████████████████████| 164/164 [00:10<00:00, 14.97it/s]
humaneval (base tests)
pass@1:	0.774
humaneval+ (base + extra tests)
pass@1:	0.732
